In [0]:
from pyspark.sql import functions as F

# Step 1: Read the TRUE raw dataset (untouched, nested spans column)
true_raw_df = spark.read.parquet("s3://ledgr-raw-data-2026/raw/")
print("Raw rows read: " + str(true_raw_df.count()))

# Step 2: Write to Bronze (idempotent: overwrite mode, since this represents 
# a full-dataset batch load, not incremental appends)
bronze_table = "ledgr.bronze.sessions_raw"
true_raw_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)

result_count = spark.table(bronze_table).count()
print("Bronze table row count: " + str(result_count))

In [0]:
# Step 3: Add Delta CHECK constraints, idempotently (safe to rerun)
def add_constraint_if_missing(table, constraint_name, constraint_sql):
    existing = [row.key for row in spark.sql("SHOW TBLPROPERTIES " + table).collect() if "constraint" in row.key.lower()]
    if not any(constraint_name in c for c in existing):
        spark.sql("ALTER TABLE " + table + " ADD CONSTRAINT " + constraint_name + " CHECK (" + constraint_sql + ")")
        print("Added constraint: " + constraint_name)
    else:
        print("Constraint already exists, skipping: " + constraint_name)

add_constraint_if_missing(bronze_table, "session_id_not_null", "session_id IS NOT NULL")
add_constraint_if_missing(bronze_table, "run_id_not_null", "run_id IS NOT NULL")
add_constraint_if_missing(
    bronze_table, "harness_known_value",
    "harness IN ('claude_code', 'openai_solo', 'smolagents_code', 'tool_calling', 'tool_calling_with_shortlisting')"
)

In [0]:
# Step 4: Fail-loud verification, so an orchestrated run fails clearly 
# if something is genuinely wrong, not silently
expected_count = 10056
if result_count != expected_count:
    raise ValueError("Bronze row count mismatch: got " + str(result_count) + ", expected " + str(expected_count))

print("Bronze ingestion verified successfully")